# Statistical Inference and Modelling — Complete Hands-On Notebook
**BITS Pilani | Course Team**

---

## Table of Contents
1. [Module 1: Basic Probability & Statistics](#module1)
2. [Module 2: Introduction to Statistical Inference](#module2)
3. [Module 3: Estimation and Hypothesis Testing](#module3)
4. [Module 4: Two-Sample Tests & Categorical Data](#module4)
5. [Module 5: Analysis of Variance (ANOVA)](#module5)
6. [Module 6: Simple Linear Regression](#module6)
7. [Module 7: Multiple Regression](#module7)
8. [Module 8: Forecasting & Time Series Analysis](#module8)
9. [Module 9: Factor Analysis](#module9)
10. [Module 10: Cluster Analysis](#module10)
11. [Module 11: Maximum Likelihood Methods](#module11)
12. [Module 12: Discrete Response Models (Logistic Regression)](#module12)

---
## Global Setup

Import all required libraries and set a global random seed for reproducibility across all modules.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import math
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.proportion import proportions_ztest
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.holtwinters import Holt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.datasets import make_blobs
from scipy.optimize import minimize

sns.set_style("whitegrid")
GLOBAL_SEED = 42
np.random.seed(GLOBAL_SEED)

print("All libraries imported successfully!")

---
<a id='module1'></a>
# Module 1: Basic Probability & Statistics

This module revisits fundamental concepts of probability, random variables, and probability distributions.

**Learning Objectives:**
- Perform exploratory data analysis (EDA) with descriptive statistics
- Understand and simulate basic probability concepts
- Explore key random variables and their distributions (Binomial, Poisson, Normal)
- Differentiate between parametric and non-parametric approaches

### 1.1 Exploratory Data Analysis & Descriptive Statistics

**Population vs. Sample**
- **Population**: The entire group you want to draw conclusions about.
- **Sample**: A subset of the population from which you collect data.

We simulate a population of 50,000 incomes using a Gamma distribution (which naturally captures the right-skewed shape of real-world income data).

In [ ]:
np.random.seed(GLOBAL_SEED)

# Simulate a skewed income population using a Gamma distribution
population_incomes = np.random.gamma(shape=5, scale=10000, size=50000)
print(f"Population size:          {len(population_incomes):,}")
print(f"Population Mean (μ):      ${np.mean(population_incomes):,.2f}")

# Draw a random sample
sample_incomes = np.random.choice(population_incomes, size=200, replace=False)
print(f"\nSample size:              {len(sample_incomes)}")
print(f"Sample Mean (x̄):          ${np.mean(sample_incomes):,.2f}")

#### 1.1.1 Measures of Central Tendency

In [ ]:
sample_mean   = np.mean(sample_incomes)
sample_median = np.median(sample_incomes)

print(f"Mean Income:   ${sample_mean:,.2f}")
print(f"Median Income: ${sample_median:,.2f}")
print(f"  → Mean > Median indicates RIGHT SKEW in the data.")

# Mode is most meaningful for categorical data
favorite_colors = np.random.choice(['Red', 'Blue', 'Green', 'Yellow'], 50,
                                    p=[0.3, 0.4, 0.2, 0.1])
mode_result = pd.Series(favorite_colors).mode()[0]
print(f"\nMode of Favorite Colors:  {mode_result}")

#### 1.1.2 Measures of Dispersion

In [ ]:
# ddof=1 applies Bessel's correction for an unbiased sample variance
sample_range    = np.max(sample_incomes) - np.min(sample_incomes)
sample_variance = np.var(sample_incomes, ddof=1)
sample_std      = np.std(sample_incomes, ddof=1)

print(f"Range:              ${sample_range:,.2f}")
print(f"Variance (s²):      {sample_variance:,.2f}")
print(f"Std Deviation (s):  ${sample_std:,.2f}")

#### 1.1.3 Visualizing the Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram with KDE
sns.histplot(sample_incomes, kde=True, bins=20, ax=axes[0])
axes[0].axvline(sample_mean,   color='r', linestyle='--', label=f'Mean (${sample_mean:,.0f})')
axes[0].axvline(sample_median, color='g', linestyle='-',  label=f'Median (${sample_median:,.0f})')
axes[0].set_title('Distribution of Sample Incomes')
axes[0].set_xlabel('Income ($)')
axes[0].set_ylabel('Frequency')
axes[0].legend()

# Box Plot
sns.boxplot(x=sample_incomes, ax=axes[1])
axes[1].set_title('Box Plot of Sample Incomes')
axes[1].set_xlabel('Income ($)')

plt.tight_layout()
plt.show()
print("The histogram confirms right skew — the mean is pulled toward the long right tail.")

### 1.2 Basic Probability Concepts

#### 1.2.1 Law of Large Numbers

As the number of trials increases, the **empirical probability** converges to the **theoretical probability**.

In [ ]:
# Small sample — noisy estimate
flips_10 = np.random.randint(0, 2, size=10)
print(f"10 flips:  {flips_10}")
print(f"P(Heads) from 10 flips:    {flips_10.mean():.4f}  (theoretical = 0.5000)")

# Large sample — converges to truth
flips_10000 = np.random.randint(0, 2, size=10000)
print(f"P(Heads) from 10,000 flips: {flips_10000.mean():.4f}  (theoretical = 0.5000)")

#### 1.2.2 Conditional Probability

$$P(L|S) = \frac{P(L \cap S)}{P(S)}$$

**Example:** Given a person smokes, what is the probability they have lung disease?

In [ ]:
Total       = 100
P_S         = 40 / Total   # P(Smoker)
P_L_and_S   = 15 / Total   # P(Lung Disease AND Smoker)
P_L_given_S = P_L_and_S / P_S

print(f"P(Smoker):                     {P_S:.3f}")
print(f"P(Lung Disease ∩ Smoker):      {P_L_and_S:.3f}")
print(f"P(Lung Disease | Smoker):      {P_L_given_S:.3f}")
print(f"\n→ 37.5% of smokers have lung disease in this dataset.")

### 1.3 Random Variables and Distributions

#### 1.3.1 Binomial Distribution (Discrete)

Models the **number of successes** in `n` independent Bernoulli trials, each with probability `p`.

$$P(X = k) = \binom{n}{k} p^k (1-p)^{n-k}$$

In [ ]:
n, p = 10, 0.5

prob_exactly_6 = stats.binom.pmf(k=6, n=n, p=p)
prob_6_or_less = stats.binom.cdf(k=6, n=n, p=p)

print(f"P(X = 6):  {prob_exactly_6:.4f}")
print(f"P(X ≤ 6):  {prob_6_or_less:.4f}")

k_values  = np.arange(0, n + 1)
pmf_values = stats.binom.pmf(k_values, n=n, p=p)

plt.figure(figsize=(8, 4))
plt.bar(k_values, pmf_values, color='steelblue', edgecolor='black')
plt.bar(6, pmf_values[6], color='tomato', edgecolor='black', label='k=6')
plt.title(f'Binomial PMF  (n={n}, p={p})')
plt.xlabel('Number of Heads (k)')
plt.ylabel('P(X = k)')
plt.xticks(k_values)
plt.legend()
plt.show()

#### 1.3.2 Poisson Distribution (Discrete)

Models the **number of events** in a fixed interval when events occur at a known average rate λ.

$$P(X = k) = \frac{e^{-\lambda} \lambda^k}{k!}$$

In [ ]:
lam = 5  # average 5 calls per hour

prob_3_calls = stats.poisson.pmf(k=3, mu=lam)
print(f"P(X = 3 calls in one hour):  {prob_3_calls:.4f}")

k_values   = np.arange(0, 15)
pmf_values = stats.poisson.pmf(k_values, mu=lam)

plt.figure(figsize=(8, 4))
plt.bar(k_values, pmf_values, color='steelblue', edgecolor='black')
plt.bar(3, pmf_values[3], color='tomato', edgecolor='black', label='k=3')
plt.title(f'Poisson PMF  (λ={lam})')
plt.xlabel('Number of Calls (k)')
plt.ylabel('P(X = k)')
plt.legend()
plt.show()

#### 1.3.3 Normal Distribution (Continuous)

The bell curve, defined by μ (mean) and σ (standard deviation). Central to the CLT.

$$f(x) = \frac{1}{\sigma\sqrt{2\pi}} e^{-\frac{1}{2}\left(\frac{x-\mu}{\sigma}\right)^2}$$

In [ ]:
mu, sigma = 70, 10  # exam scores: mean=70, std=10

prob_less_60  = stats.norm.cdf(x=60, loc=mu, scale=sigma)
prob_more_90  = 1 - stats.norm.cdf(x=90, loc=mu, scale=sigma)
prob_60_to_80 = stats.norm.cdf(80, mu, sigma) - stats.norm.cdf(60, mu, sigma)

print(f"P(X < 60):       {prob_less_60:.4f}  (~lower 16%)")
print(f"P(X > 90):       {prob_more_90:.4f}  (~upper 2.3%)")
print(f"P(60 < X < 80):  {prob_60_to_80:.4f}")

x_values  = np.linspace(mu - 4*sigma, mu + 4*sigma, 500)
pdf_values = stats.norm.pdf(x_values, loc=mu, scale=sigma)

plt.figure(figsize=(9, 4))
plt.plot(x_values, pdf_values, color='steelblue', lw=2)

# Shade P(X > 90)
x_fill = np.linspace(90, mu + 4*sigma, 200)
plt.fill_between(x_fill, stats.norm.pdf(x_fill, mu, sigma),
                 color='tomato', alpha=0.6, label=f'P(X>90) = {prob_more_90:.4f}')

plt.axvline(mu, color='gray', linestyle='--', label=f'μ = {mu}')
plt.title(f'Normal Distribution  (μ={mu}, σ={sigma})')
plt.xlabel('Score')
plt.ylabel('PDF')
plt.legend()
plt.show()

### 1.4 Parametric vs. Non-Parametric Approaches

| Approach | Assumption | Central Measure | Example Test |
|---|---|---|---|
| **Parametric** | Data follows a specific distribution (e.g., Normal) | Mean | t-test |
| **Non-Parametric** | No distributional assumptions | Median | Wilcoxon signed-rank |

For small, skewed samples, the non-parametric approach is more reliable.

In [ ]:
skewed_data = np.random.exponential(scale=10, size=20)

print(f"Mean   (Parametric focus):     {np.mean(skewed_data):.2f}")
print(f"Median (Non-Parametric focus): {np.median(skewed_data):.2f}")

# Parametric: one-sample t-test (H0: μ = 10)
t_stat, p_para = stats.ttest_1samp(skewed_data, 10)
print(f"\nParametric t-test   p-value: {p_para:.4f}")

# Non-Parametric: Wilcoxon signed-rank test (H0: median = 10)
try:
    w_stat, p_nonpara = stats.wilcoxon(skewed_data - 10)
    print(f"Non-Parametric Wilcoxon p-value: {p_nonpara:.4f}")
except ValueError as e:
    print(f"Wilcoxon test skipped: {e}")

---
<a id='module2'></a>
# Module 2: Introduction to Statistical Inference

**Learning Objectives:**
- Understand and apply sampling methods
- Define and calculate point estimates and understand their properties
- Visualize and understand the Central Limit Theorem (CLT)

### 2.1 Sampling Theory

We create a synthetic university population with three faculties having different GPA distributions.

In [ ]:
np.random.seed(GLOBAL_SEED)

data = []
data.extend([{'Faculty': 'Arts',        'GPA': np.random.normal(2.8, 0.5)} for _ in range(1000)])
data.extend([{'Faculty': 'Science',     'GPA': np.random.normal(3.2, 0.4)} for _ in range(800)])
data.extend([{'Faculty': 'Engineering', 'GPA': np.random.normal(3.5, 0.3)} for _ in range(500)])

population_df = pd.DataFrame(data)
population_df['GPA'] = population_df['GPA'].clip(0.0, 4.0)

population_mean_gpa = population_df['GPA'].mean()
print(f"Total Population:    {len(population_df):,} students")
print(f"True Population Mean GPA (μ): {population_mean_gpa:.4f}")
print(f"\nFaculty breakdown:")
print(population_df.groupby('Faculty')['GPA'].agg(['count','mean']).round(4))

In [ ]:
sample_size = 100

# Simple Random Sampling — every member has equal probability
srs_sample     = population_df.sample(n=sample_size, random_state=1)
srs_mean       = srs_sample['GPA'].mean()

# Stratified Sampling — proportional representation from each faculty
def stratified_sampler(group):
    n = max(1, round(len(group) * sample_size / len(population_df)))
    return group.sample(n=min(n, len(group)), random_state=1)

strat_sample   = population_df.groupby('Faculty', group_keys=False).apply(stratified_sampler)
strat_mean     = strat_sample['GPA'].mean()

print(f"True Population Mean:           {population_mean_gpa:.4f}")
print(f"Simple Random Sample Mean:      {srs_mean:.4f}  (error: {abs(srs_mean - population_mean_gpa):.4f})")
print(f"Stratified Sample Mean:         {strat_mean:.4f}  (error: {abs(strat_mean - population_mean_gpa):.4f})")
print("\n→ Stratified sampling often yields lower bias for heterogeneous populations.")

### 2.2 Point Estimation — Demonstrating Unbiasedness

An estimator is **unbiased** if its expected value equals the true parameter: E[x̄] = μ.
We verify this by averaging 5,000 sample means.

In [ ]:
sample_means = [population_df['GPA'].sample(n=50).mean() for _ in range(5000)]

avg_of_means = np.mean(sample_means)
print(f"True Population Mean (μ):       {population_mean_gpa:.4f}")
print(f"Average of 5,000 Sample Means:  {avg_of_means:.4f}")
print(f"Bias (E[x̄] - μ):               {avg_of_means - population_mean_gpa:.4f}")
print("\n→ Near-zero bias confirms the sample mean is an unbiased estimator.")

plt.figure(figsize=(9, 4))
sns.histplot(sample_means, bins=40, kde=True)
plt.axvline(population_mean_gpa, color='r', linestyle='--', label=f'True μ = {population_mean_gpa:.4f}')
plt.axvline(avg_of_means,        color='g', linestyle=':',  label=f'E[x̄] = {avg_of_means:.4f}')
plt.title('Sampling Distribution of the Sample Mean (n=50, 5000 samples)')
plt.xlabel('Sample Mean GPA')
plt.legend()
plt.show()

### 2.3 Central Limit Theorem (CLT)

> *The sampling distribution of the sample mean approaches Normal as n → ∞, regardless of the population's shape.*

We demonstrate this using a highly skewed Exponential population.

In [ ]:
exponential_pop = np.random.exponential(scale=10, size=10000)

def visualize_clt(population, sample_size, num_samples=1000, ax=None):
    means = [np.mean(np.random.choice(population, sample_size)) for _ in range(num_samples)]
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 3))
    sns.histplot(means, kde=True, ax=ax)
    ax.axvline(np.mean(population), color='red', linestyle='--', label='Pop. Mean')
    ax.set_title(f'Sampling Distribution  (n={sample_size})')
    ax.set_xlabel('Sample Mean')
    ax.legend()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].set_title("Population (Exponential — highly skewed)")
sns.histplot(exponential_pop, bins=50, ax=axes[0])
visualize_clt(exponential_pop, sample_size=5,  ax=axes[1])
visualize_clt(exponential_pop, sample_size=50, ax=axes[2])
plt.suptitle('Central Limit Theorem in Action', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()
print("→ As n increases from 5 to 50, the sampling distribution becomes increasingly bell-shaped.")

---
<a id='module3'></a>
# Module 3: Estimation and Hypothesis Testing

**Learning Objectives:**
- Construct and interpret Confidence Intervals
- Calculate required sample size
- Understand and conduct one-sample hypothesis tests

### 3.1 Confidence Intervals

A **Confidence Interval** gives a range of plausible values for a population parameter:

$$\text{CI} = \bar{x} \pm z_{\alpha/2} \cdot \frac{\sigma}{\sqrt{n}} \quad (\sigma \text{ known})$$

$$\text{CI} = \bar{x} \pm t_{\alpha/2, n-1} \cdot \frac{s}{\sqrt{n}} \quad (\sigma \text{ unknown})$$

In [ ]:
np.random.seed(GLOBAL_SEED)

# --- Z-interval (σ known) ---
sigma, n, x_bar = 100, 40, 780
alpha = 0.05
z_crit = stats.norm.ppf(1 - alpha/2)
moe_z  = z_crit * (sigma / math.sqrt(n))
print("--- Z-interval (σ known) ---")
print(f"Scenario: Bulb lifespan, σ=100h, n=40, x̄=780h")
print(f"z* = {z_crit:.3f},  Margin of Error = {moe_z:.2f}h")
print(f"95% CI: ({x_bar - moe_z:.2f}, {x_bar + moe_z:.2f}) hours")

print()

# --- t-interval (σ unknown) ---
n, x_bar, s = 20, 170, 8
t_crit = stats.t.ppf(1 - alpha/2, df=n-1)
moe_t  = t_crit * (s / math.sqrt(n))
print("--- t-interval (σ unknown) ---")
print(f"Scenario: Student heights, n=20, x̄=170cm, s=8cm")
print(f"t* = {t_crit:.3f},  Margin of Error = {moe_t:.2f}cm")
print(f"95% CI: ({x_bar - moe_t:.2f}, {x_bar + moe_t:.2f}) cm")

### 3.2 Sample Size Determination

$$n = \left(\frac{z_{\alpha/2} \cdot \sigma}{ME}\right)^2$$

In [ ]:
ME_desired    = 0.5   # desired margin of error (hours)
sigma_est     = 2.0   # estimated std dev
confidence    = 0.99
z_crit_99     = stats.norm.ppf(1 - (1 - confidence)/2)

n_required = ((z_crit_99 * sigma_est) / ME_desired) ** 2
print(f"Scenario: Estimate battery life, ME ≤ {ME_desired}h, σ̂ = {sigma_est}h, 99% confidence")
print(f"Required sample size: n = ⌈{n_required:.1f}⌉ = {math.ceil(n_required)}")

### 3.3 One-Sample Hypothesis Testing

**Framework:**
1. State H₀ and H₁
2. Choose significance level α
3. Compute test statistic and p-value
4. Decision: reject H₀ if p-value < α

In [ ]:
alpha = 0.05
mu_0  = 3.5   # claimed population mean GPA

# Sample drawn from a population with true mean 3.4
sample_data = np.random.normal(loc=3.4, scale=0.4, size=50)
t_stat, p_value = stats.ttest_1samp(a=sample_data, popmean=mu_0)

print(f"H₀: μ = {mu_0}  (university's claim)")
print(f"H₁: μ ≠ {mu_0}  (two-tailed)")
print(f"\nSample Mean:  {sample_data.mean():.4f}")
print(f"t-statistic:  {t_stat:.4f}")
print(f"p-value:      {p_value:.4f}")
print()
if p_value < alpha:
    print(f"Decision: REJECT H₀ (p={p_value:.4f} < α={alpha})")
    print("Evidence suggests the true average GPA differs from 3.5.")
else:
    print(f"Decision: FAIL TO REJECT H₀ (p={p_value:.4f} ≥ α={alpha})")

---
<a id='module4'></a>
# Module 4: Two-Sample Tests & Categorical Data

**Learning Objectives:**
- Perform Independent Samples t-tests (with Levene's variance check)
- Conduct two-proportion z-tests (A/B testing)
- Run a Chi-Square test for independence

In [ ]:
np.random.seed(GLOBAL_SEED)
alpha = 0.05

### 4.1 Independent Samples t-test

In [ ]:
# Simulated exam scores for two teaching methods
method_A = np.random.normal(loc=75, scale=10, size=30)
method_B = np.random.normal(loc=80, scale=10, size=30)

# Step 1: Check variance equality with Levene's test
_, p_levene = stats.levene(method_A, method_B)
equal_var   = p_levene >= alpha
print(f"Levene's Test p-value: {p_levene:.4f}  → Equal variances assumed: {equal_var}")

# Step 2: Two-sample t-test (Welch's if variances unequal)
t_stat, p_value = stats.ttest_ind(method_A, method_B, equal_var=equal_var)
print(f"\nt-statistic: {t_stat:.4f}")
print(f"p-value:     {p_value:.4f}")

if p_value < alpha:
    print("\nDecision: REJECT H₀ — significant difference in method means.")
else:
    print("\nDecision: FAIL TO REJECT H₀.")

### 4.2 Two-Proportion z-test (A/B Testing)

In [ ]:
# H₀: p_A ≥ p_B   H₁: p_A < p_B  (one-tailed — is Layout B better?)
n_A, success_A = 1000, 150   # 15% conversion
n_B, success_B = 1000, 185   # 18.5% conversion

print(f"Layout A conversion rate: {success_A/n_A:.1%}")
print(f"Layout B conversion rate: {success_B/n_B:.1%}")

z_stat, p_value = proportions_ztest(
    count=np.array([success_A, success_B]),
    nobs=np.array([n_A, n_B]),
    alternative='smaller'
)
print(f"\nz-statistic: {z_stat:.4f}")
print(f"p-value:     {p_value:.4f}")

if p_value < alpha:
    print("Decision: REJECT H₀ — Layout B has a significantly higher conversion rate.")

### 4.3 Chi-Square Test for Independence

Tests whether two categorical variables are statistically associated.

In [ ]:
# Contingency table: Education Level vs. Preferred Music Genre
observed = pd.DataFrame(
    {'High School': [30, 20, 50],
     'Bachelors':   [35, 40, 25],
     'Masters':     [25, 45, 10]},
    index=['Pop', 'Rock', 'Classical']
)
print("Observed Frequencies:")
print(observed)

chi2, p_value, dof, expected = stats.chi2_contingency(observed)

print(f"\nChi² statistic: {chi2:.4f}")
print(f"Degrees of freedom: {dof}")
print(f"p-value: {p_value:.4e}")

if p_value < alpha:
    print("\nDecision: REJECT H₀ — Education Level and Music Preference are NOT independent.")

---
<a id='module5'></a>
# Module 5: Analysis of Variance (ANOVA)

ANOVA tests whether the means of **3+ groups** differ simultaneously, controlling the family-wise error rate.

**H₀**: All group means are equal  
**H₁**: At least one group mean differs

In [ ]:
np.random.seed(GLOBAL_SEED)
alpha = 0.05
n = 20  # observations per group

# Fertilizer experiment — Fert_2 has a notably higher yield
data = pd.DataFrame({
    'Yield':      np.concatenate([
                    np.random.normal(50, 5, n),
                    np.random.normal(52, 5, n),
                    np.random.normal(60, 5, n),
                    np.random.normal(50, 5, n)]),
    'Fertilizer': ['Control']*n + ['Fert_1']*n + ['Fert_2']*n + ['Fert_3']*n
})

print("Group summary statistics:")
print(data.groupby('Fertilizer')['Yield'].agg(['count','mean','std']).round(2))

plt.figure(figsize=(9, 5))
sns.boxplot(x='Fertilizer', y='Yield', data=data, palette='Set2')
plt.title('Crop Yield by Fertilizer Type')
plt.show()

In [ ]:
# Fit OLS model and generate the ANOVA table (Type II sums of squares)
model      = ols('Yield ~ C(Fertilizer)', data=data).fit()
anova_table = sm.stats.anova_lm(model, typ=2)

print("ANOVA Table:")
print(anova_table.round(4))

p_anova = anova_table['PR(>F)'].iloc[0]
print(f"\np-value: {p_anova:.2e}")
if p_anova < alpha:
    print("Decision: REJECT H₀ — at least one fertilizer group differs significantly.")

### 5.1 Post-Hoc Testing: Tukey's HSD

ANOVA only tells us *that* a difference exists — Tukey's HSD tells us *which pairs* differ.

In [ ]:
tukey = pairwise_tukeyhsd(endog=data['Yield'], groups=data['Fertilizer'], alpha=alpha)
print(tukey)

tukey.plot_simultaneous(figsize=(9, 5))
plt.title("Tukey HSD — 95% Confidence Intervals for Group Mean Differences")
plt.show()
print("\n→ Fert_2 is significantly different from ALL other groups (non-overlapping CI).")

---
<a id='module6'></a>
# Module 6: Simple Linear Regression (SLR)

$$Y = \beta_0 + \beta_1 X + \varepsilon$$

**Learning Objectives:**
- Build and interpret an OLS regression model
- Assess model significance (R², F-test, t-tests)
- Perform residual analysis to validate assumptions

In [ ]:
np.random.seed(GLOBAL_SEED)

# True relationship: Sales = 50 + 2.5 × AdSpend + noise
ad_spend      = np.linspace(10, 100, 50) + np.random.normal(0, 10, 50)
sales_revenue = 50 + 2.5 * ad_spend + np.random.normal(0, 30, 50)

df_slr = pd.DataFrame({'Ad_Spend': ad_spend, 'Sales_Revenue': sales_revenue})

# Scatter plot
plt.figure(figsize=(8, 5))
sns.regplot(x='Ad_Spend', y='Sales_Revenue', data=df_slr, line_kws={'color':'tomato'})
plt.title('Advertising Spend vs. Sales Revenue')
plt.show()

In [ ]:
X     = sm.add_constant(df_slr['Ad_Spend'])
Y     = df_slr['Sales_Revenue']
model = sm.OLS(Y, X).fit()

print(model.summary())
print(f"\nKey Takeaways:")
print(f"  Intercept β₀ = {model.params['const']:.2f}")
print(f"  Slope     β₁ = {model.params['Ad_Spend']:.2f}  (each $1 in ad spend → ${model.params['Ad_Spend']:.2f} more sales)")
print(f"  R² = {model.rsquared:.3f} — model explains {model.rsquared*100:.1f}% of sales variance")

In [ ]:
residuals     = model.resid
fitted_values = model.fittedvalues

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Residuals vs Fitted — checks linearity & homoscedasticity
axes[0].scatter(fitted_values, residuals, alpha=0.6)
axes[0].axhline(0, color='red', linestyle='--')
axes[0].set_title('Residuals vs. Fitted Values')
axes[0].set_xlabel('Fitted Values')
axes[0].set_ylabel('Residuals')

# Q-Q Plot — checks normality of residuals
sm.qqplot(residuals, line='s', ax=axes[1])
axes[1].set_title('Q-Q Plot of Residuals')

plt.tight_layout()
plt.show()
print("→ Random scatter around 0 and Q-Q alignment indicate assumptions are met.")

---
<a id='module7'></a>
# Module 7: Multiple Regression

$$Y = \beta_0 + \beta_1 X_1 + \beta_2 X_2 + \cdots + \beta_k X_k + \varepsilon$$

**Learning Objectives:**
- Build and interpret a multiple regression model
- Use Adjusted R² for model evaluation
- Detect multicollinearity using VIF (Variance Inflation Factor)

In [ ]:
np.random.seed(GLOBAL_SEED)
n = 100

# Predictors
Size     = np.random.normal(2000, 500, n)
Bedrooms = np.random.randint(2, 6, n)
Age      = np.random.randint(1, 50, n)

# True model: Price = 50 + 0.15*Size + 10*Bedrooms - 2*Age + noise
Price = 50 + 0.15*Size + 10*Bedrooms - 2*Age + np.random.normal(0, 50, n)

df_mlr = pd.DataFrame({'Price': Price, 'Size': Size, 'Bedrooms': Bedrooms, 'Age': Age})

X = sm.add_constant(df_mlr[['Size', 'Bedrooms', 'Age']])
Y = df_mlr['Price']

model_mlr = sm.OLS(Y, X).fit()
print(model_mlr.summary())

In [ ]:
# VIF > 5 signals problematic multicollinearity; > 10 is severe
vif_data = pd.DataFrame({
    'Feature': X.columns,
    'VIF':     [variance_inflation_factor(X.values.astype(float), i) for i in range(X.shape[1])]
})

print("Variance Inflation Factors:")
print(vif_data[vif_data['Feature'] != 'const'].round(3))
print("\n→ All VIFs ≈ 1.0 → no multicollinearity among predictors.")

---
<a id='module8'></a>
# Module 8: Forecasting & Time Series Analysis

**Learning Objectives:**
- Decompose a time series into Trend, Seasonality, and Residual
- Apply Holt's Linear Trend Model (Double Exponential Smoothing)

In [ ]:
np.random.seed(GLOBAL_SEED)

# Synthetic monthly sales data with upward trend + seasonality
dates      = pd.date_range(start='2019-01-01', periods=60, freq='ME')
trend      = np.linspace(100, 300, 60)
seasonality = 50 * np.sin(np.linspace(0, 5 * 2 * np.pi, 60))
noise      = np.random.normal(0, 15, 60)
sales      = trend + seasonality + noise

ts_data = pd.Series(sales, index=dates, name='Sales')

# Decomposition: Trend + Seasonal + Residual
decomposition = seasonal_decompose(ts_data, model='additive', period=12)
fig = decomposition.plot()
fig.set_size_inches(12, 8)
fig.suptitle('Additive Decomposition of Monthly Sales', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Holt's Linear Trend model handles data with a trend component
model_holt   = Holt(ts_data, initialization_method='estimated').fit(optimized=True)
fitted_holt  = model_holt.fittedvalues
forecast_holt = model_holt.forecast(12)   # 12-month ahead forecast

print(f"Holt's Model — Estimated Parameters:")
print(f"  Smoothing level (α): {model_holt.params['smoothing_level']:.4f}")
print(f"  Smoothing trend (β): {model_holt.params['smoothing_trend']:.4f}")

plt.figure(figsize=(12, 5))
ts_data.plot(label='Actual Sales', color='steelblue')
fitted_holt.plot(label="Holt's Fitted", linestyle='--', color='purple')
forecast_holt.plot(label="12-Month Forecast", color='tomato', linewidth=2)
plt.axvline(ts_data.index[-1], color='gray', linestyle=':', label='Forecast Start')
plt.title("Holt's Linear Trend Model with 12-Month Forecast")
plt.legend()
plt.show()

---
<a id='module9'></a>
# Module 9: Factor Analysis

Factor Analysis is a **data reduction** technique that identifies underlying latent factors from a set of observed variables.

> **Note:** Requires the `factor_analyzer` library. Install with: `pip install factor_analyzer`

**Learning Objectives:**
- Apply and interpret Factor Analysis
- Use eigenvalues / Scree plot to select number of factors
- Interpret rotated factor loadings

In [ ]:
try:
    from factor_analyzer import FactorAnalyzer
    from factor_analyzer.factor_analyzer import calculate_bartlett_sphericity, calculate_kmo
    from factor_analyzer.datasets import load_bfi
    FACTOR_OK = True
    print("factor_analyzer loaded successfully.")
except ImportError:
    FACTOR_OK = False
    print("factor_analyzer not found. Install with: pip install factor_analyzer")
    print("Skipping Module 9.")

In [ ]:
if FACTOR_OK:
    # Load BFI (Big Five Inventory) dataset
    df_bfi = load_bfi().drop(['gender', 'education', 'age'], axis=1)
    df_bfi = df_bfi.apply(lambda x: x.fillna(x.median()))

    # Suitability checks
    chi2_b, p_bartlett = calculate_bartlett_sphericity(df_bfi)
    _, kmo_score       = calculate_kmo(df_bfi)
    print(f"Bartlett's Test p-value: {p_bartlett:.4e}  (want < 0.05)")
    print(f"KMO Score:               {kmo_score:.4f}    (want > 0.6)")

    # Scree Plot to choose number of factors (Kaiser criterion: eigenvalue > 1)
    fa_full = FactorAnalyzer(rotation=None, n_factors=df_bfi.shape[1])
    fa_full.fit(df_bfi)
    ev, _ = fa_full.get_eigenvalues()

    plt.figure(figsize=(10, 5))
    plt.plot(range(1, len(ev)+1), ev, marker='o')
    plt.axhline(1, color='red', linestyle='--', label='Kaiser Criterion (EV=1)')
    plt.title('Scree Plot — BFI Dataset')
    plt.xlabel('Factor Number')
    plt.ylabel('Eigenvalue')
    plt.legend()
    plt.show()

    # Fit with 5 factors and Varimax rotation
    fa5 = FactorAnalyzer(n_factors=5, rotation='varimax')
    fa5.fit(df_bfi)
    loadings = pd.DataFrame(
        fa5.loadings_,
        index=df_bfi.columns,
        columns=[f'Factor_{i+1}' for i in range(5)]
    )
    print("\nFactor Loadings (|loading| > 0.4):")
    print(loadings[abs(loadings) > 0.4].dropna(how='all').round(2))
    print("\n→ 5 factors correspond to the Big Five: N, E, C, A, O.")

---
<a id='module10'></a>
# Module 10: Cluster Analysis (K-Means)

K-Means is an **unsupervised** technique that partitions data into K clusters by minimizing within-cluster variance.

**Learning Objectives:**
- Choose optimal K using the Elbow Method
- Apply K-Means and visualize cluster assignments

In [ ]:
np.random.seed(GLOBAL_SEED)

# Generate 5 well-separated blobs
X_cluster, y_true = make_blobs(n_samples=200, centers=5, cluster_std=0.60, random_state=0)
X_scaled          = StandardScaler().fit_transform(X_cluster)

# Elbow Method — plot WCSS for K = 1..10
wcss = []
for k in range(1, 11):
    km = KMeans(n_clusters=k, init='k-means++', random_state=42, n_init=10)
    km.fit(X_scaled)
    wcss.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(range(1, 11), wcss, marker='o', color='steelblue')
plt.axvline(5, color='tomato', linestyle='--', label='Optimal K=5')
plt.title('The Elbow Method')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('WCSS (Inertia)')
plt.legend()
plt.xticks(range(1, 11))
plt.show()

In [ ]:
# Fit final model with K=5
km_final = KMeans(n_clusters=5, init='k-means++', random_state=42, n_init=10)
labels   = km_final.fit_predict(X_scaled)

sil_score = silhouette_score(X_scaled, labels)
print(f"Silhouette Score: {sil_score:.4f}  (range -1 to 1; closer to 1 is better)")

df_clusters = pd.DataFrame(X_cluster, columns=['Feature_1', 'Feature_2'])
df_clusters['Cluster'] = labels

plt.figure(figsize=(9, 6))
sns.scatterplot(x='Feature_1', y='Feature_2', hue='Cluster',
                data=df_clusters, palette='Set1', s=80)
plt.scatter(*km_final.cluster_centers_.T[::-1],
            marker='X', s=200, color='black', label='Centroids', zorder=5)

# Note: centroids are in scaled space; approximate for visualization
plt.title('K-Means Clustering (K=5)')
plt.legend(title='Cluster')
plt.show()

---
<a id='module11'></a>
# Module 11: Maximum Likelihood Estimation (MLE)

MLE finds the parameter values that **maximize the probability** of observing the data.

$$\hat{\theta}_{MLE} = \arg\max_\theta \mathcal{L}(\theta \mid \mathbf{x})$$

In practice we minimize the **negative log-likelihood (NLL)**.

In [ ]:
np.random.seed(GLOBAL_SEED)

# Example: 10 coin flips, 7 heads observed — what is the MLE of p?
n_flips, k_heads = 10, 7
p_values    = np.linspace(0.01, 0.99, 200)
likelihoods = stats.binom.pmf(k_heads, n_flips, p_values)

mle_p = p_values[np.argmax(likelihoods)]
print(f"Analytical MLE: p̂ = k/n = {k_heads}/{n_flips} = {k_heads/n_flips}")
print(f"Numerical MLE:  p̂ ≈ {mle_p:.3f}")

plt.figure(figsize=(8, 4))
plt.plot(p_values, likelihoods, color='steelblue')
plt.axvline(mle_p, color='tomato', linestyle='--', label=f'MLE p̂ = {mle_p:.2f}')
plt.title(f'Likelihood Function  L(p | k={k_heads}, n={n_flips})')
plt.xlabel('p (Probability of Heads)')
plt.ylabel('Likelihood')
plt.legend()
plt.show()

In [ ]:
# MLE for Normal distribution via numerical minimisation of NLL
data_normal = np.random.normal(loc=50, scale=10, size=100)

def normal_nll(params, data):
    mu, sigma = params
    if sigma <= 0:
        return np.inf
    return -np.sum(stats.norm.logpdf(data, loc=mu, scale=sigma))

result    = minimize(normal_nll, x0=[40, 5], args=(data_normal,),
                     method='L-BFGS-B', bounds=[(None, None), (1e-6, None)])
mle_mu, mle_sigma = result.x

print(f"True parameters:   μ=50, σ=10")
print(f"MLE estimates:     μ̂={mle_mu:.4f}, σ̂={mle_sigma:.4f}")
print(f"\n→ Note: MLE for σ uses n in the denominator (biased), hence slightly < sample std.")
print(f"  Sample std (unbiased): {np.std(data_normal, ddof=1):.4f}")

---
<a id='module12'></a>
# Module 12: Discrete Response Models — Logistic Regression

When the outcome is **binary** (0/1), linear regression is inappropriate. Logistic regression models the **log-odds**:

$$\log\left(\frac{p}{1-p}\right) = \beta_0 + \beta_1 X$$

The predicted probability follows an S-shaped curve bounded in [0, 1]:

$$P(Y=1|X) = \frac{1}{1 + e^{-(\beta_0 + \beta_1 X)}}$$

In [ ]:
np.random.seed(GLOBAL_SEED)

# Simulate admission data based on GPA
GPA         = np.random.uniform(2.0, 4.0, 100)
probability = 1 / (1 + np.exp(-(-10 + 3.5 * GPA)))
Admitted    = np.random.binomial(1, probability)

df_logit = pd.DataFrame({'GPA': GPA, 'Admitted': Admitted})
print(f"Admitted: {Admitted.sum()}/100   Not Admitted: {(1-Admitted).sum()}/100")

plt.figure(figsize=(8, 5))
sns.regplot(x='GPA', y='Admitted', data=df_logit, logistic=True, ci=None,
            scatter_kws={'alpha': 0.5}, line_kws={'color': 'tomato', 'lw': 2})
plt.title('Logistic Regression — Probability of Admission vs. GPA')
plt.xlabel('GPA')
plt.ylabel('P(Admitted)')
plt.show()

In [ ]:
X_logit = sm.add_constant(df_logit['GPA'])
Y_logit = df_logit['Admitted']

model_logit = sm.Logit(Y_logit, X_logit).fit()
print(model_logit.summary())

# Odds Ratios = exp(coefficient)
odds_ratios = np.exp(model_logit.params)
print("\nOdds Ratios:")
print(odds_ratios.round(3))

or_gpa = odds_ratios['GPA']
print(f"\n→ For each 1-point increase in GPA, the odds of admission multiply by {or_gpa:.2f}x.")
print(f"  A student with GPA 3.5 is roughly {or_gpa:.0f}x more likely to be admitted than GPA 2.5.")

---
## Summary

| Module | Topic | Key Technique |
|--------|-------|---------------|
| 1 | Probability & Distributions | Binomial, Poisson, Normal |
| 2 | Statistical Inference | CLT, Sampling, Point Estimation |
| 3 | Estimation & Hypothesis Testing | Confidence Intervals, t-test |
| 4 | Two-Sample & Categorical Tests | t-test, A/B, Chi-Square |
| 5 | ANOVA | F-test, Tukey's HSD |
| 6 | Simple Linear Regression | OLS, Residual Analysis |
| 7 | Multiple Regression | Adjusted R², VIF |
| 8 | Time Series | Decomposition, Holt's Model |
| 9 | Factor Analysis | Eigenvalues, Varimax Rotation |
| 10 | Cluster Analysis | K-Means, Elbow Method |
| 11 | Maximum Likelihood | NLL Minimisation |
| 12 | Logistic Regression | Log-Odds, Odds Ratios |